In [ ]:
from pyspark import SparkConf, SparkContext

conf = SparkConf().setMaster("local[*]").setAppName("MyApp")
sc = SparkContext(conf=conf)

lines = sc.textFile("ml-100k/u.data")

ratings = lines.map(lambda x: x.split()[2])
ret = ratings.countByValue()

for rating, count in sorted(ret.items(), key=lambda x: x[1], reverse=True):
    print(f"{rating}: {count}")

sc.stop()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/29 14:21:47 WARN Utils: Your hostname, Laptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/29 14:21:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 14:21:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4: 34174
3: 27145
5: 21201
2: 11370
1: 6110


In [4]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable

conf = SparkConf().setMaster("local[*]").setAppName("MinTemps")
sc = SparkContext(conf=conf)

def parse_line(line):
    fields = line.split(",")
    station_id = fields[0]
    entry_type = fields[2]
    temp = float(fields[3])
    return (station_id, entry_type, temp)

lines = sc.textFile('data/weatherData1800s.csv')
parsed_lines = lines.map(parse_line)
max_temps = parsed_lines.filter(lambda x: 'TMAX' in x[1])
min_temps = parsed_lines.filter(lambda x: 'TMIN' in x[1])
station_temps = max_temps.map(lambda x: (x[0], x[2]))
max_temps = station_temps.reduceByKey(lambda x, y: max(x, y))
min_temps = station_temps.reduceByKey(lambda x, y: min(x, y))
ret = max_temps.collect()

table = PrettyTable()
table.field_names = ["Station ID", "Temperature"]
for id, temp in ret:
    table.add_row([id, temp])

print(table)
sc.stop()


+-------------+-------------+
|  Station ID | Temperature |
+-------------+-------------+
| ITE00100554 |    323.0    |
| EZE00100082 |    323.0    |
+-------------+-------------+


In [ ]:
from pyspark import SparkContext

sc = SparkContext("local[*]", "DataIO")

# Load the CSV file
lines = sc.textFile("sales_data.csv")

# Skip header line
header = lines.first()
data = lines.filter(lambda line: line != header)

print(f"Header: {header}")
print(f"Data records: {data.count()}")
print(f"First record: {data.first()}")

def parse_record(line):
    """Parse CSV line into structured data."""
    parts = line.split(",")
    return {
        "product_id": parts[0],
        "name": parts[1],
        "category": parts[2],
        "price": float(parts[3]),
        "quantity": int(parts[4])
    }

parsed = data.map(parse_record)

# Show parsed data
for record in parsed.take(3):
    print(record)